In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle

In [7]:
# Generate synthetic training data
def generate_training_data(n_samples=5000):
    """Generate synthetic loan application data"""
    np.random.seed(42)
    
    data = {
        'monthly_income': np.random.normal(35000, 15000, n_samples).clip(10000, 100000),
        'monthly_expenses': np.random.normal(20000, 10000, n_samples).clip(5000, 60000),
        'savings_ratio': np.random.uniform(0, 0.6, n_samples),
        'financial_stability_score': np.random.uniform(0.2, 1.0, n_samples),
        'cash_flow_stability': np.random.uniform(0.3, 1.0, n_samples),
        'credit_score': np.random.normal(680, 80, n_samples).clip(300, 850),
        'existing_loans': np.random.normal(50000, 30000, n_samples).clip(0, 200000),
        'account_balance': np.random.normal(40000, 25000, n_samples).clip(1000, 150000),
        'transaction_frequency': np.random.randint(20, 100, n_samples),
    }
    
    df = pd.DataFrame(data)
    
    # Generate target variable (loan approval)
    df['loan_approved'] = (
        (df['savings_ratio'] > 0.2) &
        (df['financial_stability_score'] > 0.5) &
        (df['credit_score'] > 600) &
        (df['monthly_income'] > 25000)
    ).astype(int)
    
    # Add some noise
    noise_indices = np.random.choice(df.index, size=int(0.1 * n_samples), replace=False)
    df.loc[noise_indices, 'loan_approved'] = 1 - df.loc[noise_indices, 'loan_approved']
    
    return df

In [3]:
generate_training_data()

,monthly_income,monthly_expenses,savings_ratio,financial_stability_score,cash_flow_stability,credit_score,existing_loans,account_balance,transaction_frequency,loan_approved
0,42450.712295,15762.403180,0.192753,0.611557,0.420057,631.413847,38541.311374,78749.105675,47,0
1,32926.035482,15465.858916,0.061812,0.655593,0.542762,632.580502,72524.307166,63446.806434,71,0
2,44715.328072,5000.000000,0.043085,0.214980,0.979071,755.339908,35125.332895,56537.880136,60,1
3,57845.447846,16699.098083,0.056566,0.768982,0.846551,722.260861,79264.986778,32654.633350,40,0
4,31487.699379,27328.290818,0.349722,0.769915,0.497814,752.297469,89149.173225,60737.189746,61,1
...,...,...,...,...,...,...,...,...,...,...
4995,34265.524537,33011.020630,0.594231,0.746350,0.392481,775.291470,28797.510231,7673.035138,41,1
4996,45671.158692,5000.000000,0.390107,0.879929,0.466063,667.905786,32369.421389,45475.403454,88,0
4997,81693.653016,12946.832761,0.076272,0.372579,0.749923,720.675768,38412.651455,29904.147707,40,0
4998,47120.542841,24957.655730,0.303702,0.803061,0.336313,719.520133,70540.480628,32320.069058,75,1


In [9]:
def train_model():
    """Train Random Forest model for loan prediction"""
    print("Generating training data...")
    df = generate_training_data()
    
    # Features and target
    X = df.drop('loan_approved', axis=1)
    y = df['loan_approved']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Random Forest
    print("Training Random Forest model...")
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test_scaled)
    
    print("\n=== Model Performance ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n=== Feature Importance ===")
    print(feature_importance)
    
    # Save model and scaler
    print("\nSaving model and scaler...")
    with open('backend/ml/loan_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    
    with open('backend/ml/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    
    print("Model training complete!")
    return model, scaler

if __name__ == "__main__":
    train_model()


Generating training data...
Training Random Forest model...

=== Model Performance ===
Accuracy: 0.8880
Precision: 0.8964
Recall: 0.7515
F1 Score: 0.8176

Confusion Matrix:
[[637  29]
 [ 83 251]]

=== Feature Importance ===
                     feature  importance
3  financial_stability_score    0.282419
2              savings_ratio    0.233866
0             monthly_income    0.203514
5               credit_score    0.160635
4        cash_flow_stability    0.027428
7            account_balance    0.025876
1           monthly_expenses    0.024974
6             existing_loans    0.021697
8      transaction_frequency    0.019590

Saving model and scaler...
Model training complete!


In [10]:

# Load trained model
with open("backend/ml/loan_model.pkl", "rb") as f:
    model = pickle.load(f)

# Load scaler
with open("backend/ml/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)


def predict_loan(features):
    """Predict loan approval"""

    features = np.array([features])

    # Scale input
    features_scaled = scaler.transform(features)

    # Prediction
    prediction = model.predict(features_scaled)[0]
    probability = model.predict_proba(features_scaled)[0][1]

    # Risk level
    if probability > 0.8:
        risk = "Low Risk"
    elif probability > 0.6:
        risk = "Medium Risk"
    else:
        risk = "High Risk"

    print("\n=== Prediction Result ===")

    if prediction == 1:
        print("Loan Approved")
    else:
        print("Loan Rejected")

    print("Approval Probability:", round(probability, 3))
    print("Risk Level:", risk)


# -------------------------------
# Sample Inputs for Testing
# -------------------------------

sample_inputs = {

    "Strong Applicant": [
        50000,   # monthly_income
        20000,   # monthly_expenses
        0.35,    # savings_ratio
        0.75,    # financial_stability_score
        0.85,    # cash_flow_stability
        720,     # credit_score
        20000,   # existing_loans
        60000,   # account_balance
        70       # transaction_frequency
    ],

    "Medium Applicant": [
        30000,
        22000,
        0.18,
        0.55,
        0.60,
        640,
        50000,
        25000,
        45
    ],

    "Weak Applicant": [
        18000,
        16000,
        0.05,
        0.30,
        0.40,
        520,
        90000,
        5000,
        25
    ]
}


if __name__ == "__main__":

    print("=== Loan Prediction Test ===\n")

    print("Choose test option:")
    print("1 - Manual Input")
    print("2 - Run Sample Test Cases")

    choice = input("Enter choice: ")

    # --------------------------------
    # Manual Input
    # --------------------------------
    if choice == "1":

        monthly_income = float(input("Monthly Income: "))
        monthly_expenses = float(input("Monthly Expenses: "))
        savings_ratio = float(input("Savings Ratio (0-1): "))
        financial_stability_score = float(input("Financial Stability Score (0-1): "))
        cash_flow_stability = float(input("Cash Flow Stability (0-1): "))
        credit_score = float(input("Credit Score (300-850): "))
        existing_loans = float(input("Existing Loans Amount: "))
        account_balance = float(input("Account Balance: "))
        transaction_frequency = float(input("Transaction Frequency: "))

        features = [
            monthly_income,
            monthly_expenses,
            savings_ratio,
            financial_stability_score,
            cash_flow_stability,
            credit_score,
            existing_loans,
            account_balance,
            transaction_frequency
        ]

        predict_loan(features)

    # --------------------------------
    # Sample Tests
    # --------------------------------
    elif choice == "2":

        for name, features in sample_inputs.items():
            print("\n=========================")
            print("Test Case:", name)
            print("=========================")

            predict_loan(features)

    else:
        print("Invalid option")

=== Loan Prediction Test ===

Choose test option:
1 - Manual Input
2 - Run Sample Test Cases


Enter choice:  2



Test Case: Strong Applicant

=== Prediction Result ===
Loan Approved
Approval Probability: 0.841
Risk Level: Low Risk

Test Case: Medium Applicant

=== Prediction Result ===
Loan Rejected
Approval Probability: 0.116
Risk Level: High Risk

Test Case: Weak Applicant

=== Prediction Result ===
Loan Rejected
Approval Probability: 0.186
Risk Level: High Risk


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
